# Notebook 1: Capital Market Data Platform - Data Ingestion
This notebook reads stock/ETF CSVs, enriches them, merges metadata, and saves outputs.

In [ ]:
# Install once if needed
# !pip install pandas pyarrow sqlalchemy psycopg2-binary

import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine

# ---------------- USER CONFIGURATION ----------------
# Update these paths to match your folders.

STOCK_FOLDER = Path("data/stocks")
ETF_FOLDER = Path("data/etfs")
META_FILE = Path("data/metadata/symbols_valid_meta.csv")
OUTPUT_FOLDER = Path("processed")
OUTPUT_FOLDER.mkdir(exist_ok=True)

print("Configuration complete.")


## Read all Stock and ETF CSV files

In [ ]:
def load_market_folder(folder_path: Path, asset_type: str):
    '''
    Reads every CSV in the folder and returns one combined DataFrame.

    asset_type:
        Stock or ETF
    '''
    all_frames = []

    # Find every CSV file
    csv_files = list(folder_path.glob("*.csv"))
    print(f"Found {len(csv_files)} {asset_type} files")

    for file in csv_files:

        # Read CSV
        df = pd.read_csv(file)

        # File name becomes ticker symbol
        ticker = file.stem.upper()

        # Add additional columns
        df["Symbol"] = ticker
        df["Asset_Type"] = asset_type
        df["Source_File"] = file.name

        all_frames.append(df)

    if len(all_frames)==0:
        return pd.DataFrame()

    return pd.concat(all_frames, ignore_index=True)

stocks_df = load_market_folder(STOCK_FOLDER,"Stock")
etfs_df = load_market_folder(ETF_FOLDER,"ETF")

print("Stocks:",len(stocks_df))
print("ETFs:",len(etfs_df))


## Combine all market data

In [ ]:
market_df = pd.concat([stocks_df, etfs_df], ignore_index=True)

# Convert columns to correct data types
market_df["Date"] = pd.to_datetime(market_df["Date"], errors="coerce")

for col in ["Open","High","Low","Close","Adj Close","Volume"]:
    if col in market_df.columns:
        market_df[col] = pd.to_numeric(market_df[col], errors="coerce")

print(market_df.head())
print(market_df.shape)


## Merge metadata

In [ ]:
meta_df = pd.read_csv(META_FILE)

master_df = market_df.merge(
    meta_df,
    how="left",
    on="Symbol"
)

print(master_df.head())
print(master_df.info())


## Save outputs

In [ ]:
csv_file = OUTPUT_FOLDER / "master_market_data.csv"
parquet_file = OUTPUT_FOLDER / "master_market_data.parquet"

master_df.to_csv(csv_file,index=False)
master_df.to_parquet(parquet_file,index=False)

print("CSV saved:",csv_file)
print("Parquet saved:",parquet_file)

# ---------- OPTIONAL PostgreSQL ----------
# Update connection string before using.
POSTGRES_URL = ""
if POSTGRES_URL:
    engine = create_engine(POSTGRES_URL)
    master_df.to_sql(
        "master_market_data",
        engine,
        if_exists="replace",
        index=False,
        chunksize=10000,
        method="multi"
    )
    print("Loaded into PostgreSQL")
else:
    print("PostgreSQL step skipped.")


# What you learned

- Read thousands of CSV files.
- Added Symbol, Asset_Type, Source_File.
- Combined Stock and ETF data.
- Merged instrument metadata.
- Saved data as CSV and Parquet.
- Optional PostgreSQL loading.

Next notebook: Data Quality Engine.
